In [2]:
# Importar las librerÃ­as necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

In [3]:
# URL del archivo en el bucket público de AWS
url = "https://hybridge-education-machine-learning-datasets.s3.us-east-1.amazonaws.com/Fraud.csv"
df = pd.read_csv(url)

In [4]:
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [5]:
len(df)

6362620

# ¡¡¡6,362,620 OBSERVACIONES!!!

## Actividad 1: Exploración Inicial

En esta sección cargamos los datos y realizamos una revisión básica de su estructura.

In [6]:
# Revisar información general de las columnas y tipos de datos
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            str    
 2   amount          float64
 3   nameOrig        str    
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        str    
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), str(3)
memory usage: 534.0 MB


In [7]:
# Verificar si hay valores nulos en el dataset
df.isnull().sum()

step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

## Actividad 2: Preprocesamiento de Datos

Para poder entrenar nuestro modelo de regresión logística, debemos realizar los siguientes pasos:
1. Eliminar columnas que no aportan información relevante (como nombres de origen y destino).
2. Codificar variables categóricas si es necesario.
3. Escalar las características numéricas.

In [8]:
# 1. Eliminar columnas irrelevantes
df_clean = df.drop(['nameOrig', 'nameDest', 'isFlaggedFraud'], axis=1)

# 2. Codificación de variables categóricas (columna 'type')
df_clean = pd.get_dummies(df_clean, columns=['type'], drop_first=True)

# Mostrar las primeras filas del dataframe procesado
df_clean.head()

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER
0,1,9839.64,170136.0,160296.36,0.0,0.0,0,False,False,True,False
1,1,1864.28,21249.0,19384.72,0.0,0.0,0,False,False,True,False
2,1,181.00,181.0,0.00,0.0,0.0,1,False,False,False,True
3,1,181.00,181.0,0.00,21182.0,0.0,1,True,False,False,False
4,1,11668.14,41554.0,29885.86,0.0,0.0,0,False,False,True,False


In [9]:
# 3. División de datos en entrenamiento y prueba
X = df_clean.drop('isFraud', axis=1)
y = df_clean['isFraud']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# 4. Escalamiento de características (StandardScaler)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Tamaño de entrenamiento: {X_train_scaled.shape}")
print(f"Tamaño de prueba: {X_test_scaled.shape}")

Tamaño de entrenamiento: (4453834, 10)
Tamaño de prueba: (1908786, 10)


## Actividad 3: Entrenamiento del Modelo

Utilizaremos un modelo de Regresión Logística para clasificar las transacciones. Dado el desbalance de clases, esto es un punto clave a observar en la evaluación posterior.

In [10]:
# Inicializar y entrenar el modelo de Regresión Logística
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

print("Entrenamiento completado exitosamente.")

Entrenamiento completado exitosamente.


## Actividad 4: Evaluación del Modelo

Probamos el modelo con el conjunto de prueba y analizamos métricas como Precisión, Recall y F1-Score.

In [11]:
# Realizar predicciones
y_pred = model.predict(X_test_scaled)

# Mostrar reporte de clasificación
print("Reporte de Clasificación:")
print(classification_report(y_test, y_pred))

# Mostrar matriz de confusión
print("Matriz de Confusión:")
print(confusion_matrix(y_test, y_pred))

Reporte de Clasificación:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1906322
           1       0.92      0.39      0.54      2464

    accuracy                           1.00   1908786
   macro avg       0.96      0.69      0.77   1908786
weighted avg       1.00      1.00      1.00   1908786

Matriz de Confusión:
[[1906234      88]
 [   1514     950]]


## Actividad 5: Conclusiones y Preguntas

Responde a las siguientes preguntas basadas en los resultados obtenidos.

### 1. ¿Cómo afecta el desbalance de clases al desempeño del modelo?

In [ ]:
# El desbalance de clases (0.0013% de fraude) hace que la precisión global sea muy alta,
# pero el recall para la clase 1 (fraude) suele ser bajo si no se maneja el desbalance.
# El modelo tiende a predecir la clase mayoritaria por defecto.

### 2. ¿Qué métrica es más importante para este problema y por qué?

In [ ]:
# En detección de fraude, el Recall es crítico porque queremos detectar la mayor cantidad
# de fraudes posibles (minimizar falsos negativos), aunque esto implique algunos falsos positivos.